# 表格資料 ML：XGBoost/隨機森林良率預測與特徵重要性

- 目標：掌握表格資料（Tabular Data）的特徵工程技巧。使用 scikit-learn 與 XGBoost 建立良率預測模型，透過交叉驗證（k-Fold Cross Validation）確保穩健性，並利用 Optuna 進行全自動超參數調優，最後輸出特徵重要性（Feature Importance）以解釋模型判斷的物理意義。


## 1. 特徵工程（MES/NPI 原始資料提取）

- 實戰場景：原始的生產資料包含了類別型（如機台 Tool ID、產品型號）與連續型（如頻寬、溫度、針壓）特徵。我們需要對類別型欄位進行獨熱編碼（One-Hot Encoding），並處理可能導致模型偏誤的資料。


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
import optuna

# 1. 模擬 NPI 與 MES 整併後的原始特徵資料集 (500 筆晶圓紀錄)
np.random.seed(42)
num_samples = 500

raw_data = {
    "tool_id": np.random.choice(["TOOL_01", "TOOL_02", "TOOL_03"], size=num_samples),
    "recipe_ver": np.random.choice(["V1.0", "V2.0"], size=num_samples),
    "avg_temp_c": np.random.normal(loc=45.0, scale=1.2, size=num_samples),
    "max_force_g": np.random.normal(loc=120.0, scale=5.0, size=num_samples),
    "bandwidth_ghz": np.random.normal(loc=28.0, scale=0.8, size=num_samples),
}

df_features = pd.DataFrame(raw_data)

# 模擬真實的物理因果：如果溫度過高且頻寬過低，良率就會大幅下滑 (當作預測目標)
df_features["actual_yield"] = (
    98.0
    - (df_features["avg_temp_c"] - 45.0) * 1.5
    + (df_features["bandwidth_ghz"] - 28.0) * 2.0
    - (df_features["max_force_g"] > 125).astype(int) * 3.0
    + np.random.normal(0, 0.5, size=num_samples)
)
# 限制良率在 0~100% 之間
df_features["actual_yield"] = df_features["actual_yield"].clip(0, 100)

# 2. 執行特徵工程：對類別型特徵進行 One-Hot Encoding
print(">>> 執行特徵工程：類別型欄位轉化...")
df_processed = pd.get_dummies(
    df_features, columns=["tool_id", "recipe_ver"], drop_first=True
)

# 拆分特徵矩陣 X 與 目標變數 y
X = df_processed.drop(columns=["actual_yield"])
y = df_processed["actual_yield"]

print(f"✅ 特徵工程完成。特徵維度: {X.shape} | 範例欄位: {list(X.columns[:5])}")


## 2. Optuna 自動化超參數調優與 k-Fold 交叉驗證

- 實戰場景：手動調整 XGBoost 的樹深度（max_depth）或學習率（learning_rate）非常沒效率。我們使用 Optuna 建立一個自動優化機制，並在內部結合 5-Fold 交叉驗證，防止模型過擬合（Overfitting）。


In [ ]:
# 隱藏 Optuna 預設的繁雜日誌
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    """定義 Optuna 的調優目標函式"""
    # 1. 定義超參數搜尋空間 (Hyperparameter Search Space)
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    # 2. 設定 5-Fold 交叉驗證
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # 訓練模型
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)

        # 預測並計算驗證集均方誤差 (MSE)
        preds = model.predict(X_val)
        mse = mean_squared_error(y_val, preds)
        scores.append(mse)

    # 回傳 5 摺的平均 MSE，Optuna 會自動將其最小化
    return np.mean(scores)


# 3. 啟動 Optuna 尋找最佳參數
print(">>> 啟動 Optuna 自動化參數調優 Pipeline（預估執行數秒）...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("=" * 50)
print(f"🎯 最佳化完成！最低驗證集平均 MSE: {study.best_value:.4f}")
print("📦 最佳超參數組合:")
for k, v in study.best_params.items():
    print(f"   - {k}: {v}")
print("=" * 50)


## 3. 模型最終訓練與特徵重要性 (Feature Importance)

- 實戰場景：拿到最佳參數後，我們用全部資料訓練一個最終的 XGBoost 模型，並將隨機森林（Random Forest）作為對照組。最後輸出特徵重要性，這在面試與產線檢討時至關重要，能直接指出影響良率的關鍵物理變數。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 使用 Optuna 找到的最佳參數訓練最終的 XGBoost 模型
best_xgb = xgb.XGBRegressor(**study.best_params, random_state=42)
best_xgb.fit(X, y)

# 2. 同步訓練隨機森林作為 Baseline 對照組
rf_model = RandomForestRegressor(
    n_estimators=100, max_depth=6, random_state=42, n_jobs=-1
)
rf_model.fit(X, y)

# 3. 提取特徵重要性並建立 DataFrame
importance_df = pd.DataFrame(
    {
        "Feature": X.columns,
        "XGBoost_Importance": best_xgb.feature_importances_,
        "RandomForest_Importance": rf_model.feature_importances_,
    }
).sort_values(by="XGBoost_Importance", ascending=False)

print("\n📊 特徵重要性分析對照表：")
print(importance_df.to_string(index=False))

# 4. 繪製特徵重要性條形圖
plt.figure(figsize=(8, 4))
sns.barplot(x="XGBoost_Importance", y="Feature", data=importance_df, palette="viridis")
plt.title("XGBoost 模型預測良率：關鍵參數重要性分析 (Feature Importance)", fontsize=12)
plt.xlabel("特徵重要性權重")
plt.ylabel("製程/測試參數名稱")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


- 總結：在我的專案中，預測晶圓良率不能只追求高精準度，更必須提供合理的『物理意義』供製程工程師參考。我建立了完整的機器學習 Pipeline：首先，我提取 MES 與 NPI 的原始資料，將 Tool ID 等欄位進行 One-Hot 編碼轉換。為了防範模型產生過擬合（Overfitting），我設計了 5-Fold 交叉驗證機制，並引入 Optuna 自動化超參數調優引擎，在短短幾十次 Trial 內就能自動尋找到 XGBoost 最佳的樹深度與學習率。最終，我透過模型輸出的 特徵重要性 (Feature Importance) 發現，測試時的 avg_temp_c（探針床平均溫度）與 bandwidth_ghz（頻寬）對良率有決定性的影響。這項數據分析結果能直接轉換為實際的產線決策，協助硬體團隊調整機台的冷卻配方，這正是結合資料科學與領域知識（Domain Knowledge）的核心價值。
